# DQMBot — Batch Image Query Driver

**Image layout expected:**
```
images/
    <plotName>/
        <plotName>_run<XXXXXX>.png
```

**Output layout (with run_id — preserves previous runs):**
```
results/
    <run_id>/
        <plotName>/
            <plotName>_<model>_run<XXXXXX>.txt
        summary_<run_id>.csv
```

**Output layout (no run_id — overwrites):**
```
results/
    <plotName>/
        <plotName>_<model>_run<XXXXXX>.txt
    summary.csv
```

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from owui_client import (
    list_models, get_knowledge_map, query,
    batch_query_images, _collect_images, resolve_output_dir, resolve_output_file,
)

print('owui_client loaded OK')

owui_client loaded OK


In [2]:
# ── Discover available models and knowledge collections ───────────────────────
print('=== Models ===')
for m in list_models():
    print(' ', m)

print()
print('=== Knowledge collections ===')
kb_map = get_knowledge_map()
for name, kid in kb_map.items():
    print(f'  {name:30s}  {kid}')

=== Models ===


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [7]:
# ── Configuration ─────────────────────────────────────────────────────────────

IMAGE_ROOT  = Path('images')
OUTPUT_ROOT = Path('results')

# Set a string to preserve previous runs alongside this one.
# Leave as None to overwrite.
RUN_ID = 'baseline'
# RUN_ID = None

MODELS = [
 'qwen2.5vl:latest',            #7b    
 'qwen2.5vl:32b',               #32b
 'qwen3-vl:latest',             #8b
 'litellm-ow.qwen/qwen3.6',     #35b 
 'gemma3:latest',               #4b
 'litellm-ow.google/gemma4-31b',#31b
]


COLLECTIONS = [
    kb_map['DQM shift rules'],
]

SYSTEM_PROMPT = (
 """\
You are an assistant to shifters of the CMS experiment during detector operations.
You are tasked to judge if a input plot is good or bad.
 
In your output, make 4 sections:
 - Quote the relevant section of instructions for the input plot
 - Describe the input plot
 - Compare input plot to the instruction
 - Decide if the plot is good or bad\
"""
)

PROMPT = (
    ''
)

DELAY = 1.5
# ──────────────────────────────────────────────────────────────────────────────

In [8]:
# ── Sanity check: show what will be processed and where it will land ──────────
pairs = _collect_images(IMAGE_ROOT, ('.png', '.jpg', '.jpeg', '.webp'))
plot_names = sorted(set(p for p, _ in pairs))

print(f'Plots found  : {len(plot_names)}')
for pn in plot_names:
    imgs = [img for p, img in pairs if p == pn]
    print(f'  {pn}/  ({len(imgs)} images)')
    for img in imgs:
        print(f'    {img.name}')

print(f'\nModels       : {len(MODELS)}')
for m in MODELS:
    print(f'  {m}')

print(f'\nRun ID       : {RUN_ID or "(none — overwrite mode)"}')
print(f'Total queries: {len(pairs) * len(MODELS)}')

print('\nExample output paths:')
for model in MODELS:
    plot_name, img = pairs[0]
    d = resolve_output_dir(OUTPUT_ROOT, plot_name, RUN_ID)
    f = resolve_output_file(d, img, model)
    print(f'  {f}')

Plots found  : 1
  ecalOccRecdEtWgt/  (1 images)
    ecalOccRecdEtWgt_run398185.png

Models       : 6
  qwen2.5vl:latest
  qwen2.5vl:32b
  qwen3-vl:latest
  litellm-ow.qwen/qwen3.6
  gemma3:latest
  litellm-ow.google/gemma4-31b

Run ID       : baseline
Total queries: 6

Example output paths:
  results/baseline/ecalOccRecdEtWgt/ecalOccRecdEtWgt_run398185_qwen2.5vl_latest.txt
  results/baseline/ecalOccRecdEtWgt/ecalOccRecdEtWgt_run398185_qwen2.5vl_32b.txt
  results/baseline/ecalOccRecdEtWgt/ecalOccRecdEtWgt_run398185_qwen3-vl_latest.txt
  results/baseline/ecalOccRecdEtWgt/ecalOccRecdEtWgt_run398185_litellm-ow.qwen_qwen3.6.txt
  results/baseline/ecalOccRecdEtWgt/ecalOccRecdEtWgt_run398185_gemma3_latest.txt
  results/baseline/ecalOccRecdEtWgt/ecalOccRecdEtWgt_run398185_litellm-ow.google_gemma4-31b.txt


In [9]:
# ── Smoke test: one image, first model ───────────────────────────────────────
if pairs:
    plot_name, img = pairs[0]
    test = query(
        PROMPT,
        model=MODELS[2],
        system=SYSTEM_PROMPT,
        image_path=img,
        collection_ids=COLLECTIONS,
    )
    print(f"Plot    : {plot_name}")
    print(f"Model   : {test['model_used']}")
    print(f"Image   : {test['image']}")
    print(f"Latency : {test['latency_s']}s")
    print(f"Error   : {test['error']}")
    print()
    print(test['response'])

Plot    : ecalOccRecdEtWgt
Model   : qwen3-vl:latest
Image   : images/ecalOccRecdEtWgt/ecalOccRecdEtWgt_run398185.png
Latency : 89.49s
Error   : None

- Quote the relevant section of instructions for the input plot  
The CMS plotting guidelines require plots to include **explicit labels for all data elements** (e.g., marker types or anomalies) and ensure **clear axis labeling with physical units**. Unexplained visual artifacts (such as white squares without associated legends) are explicitly prohibited, as they compromise interpretation during shift operations.

- Describe the input plot  
The plot displays "ECal TP ET-weighted Occupancy at Layer1" using a heatmap of `iPhi` vs. `iEta`. The color scale quantifies entries (total = 5.85254e+07), with red representing high occupancy (>80k) and blue low occupancy (~0). Unexplained white square markers are overlaid across the plot.

- Compare input plot to the instruction  
1. **Missing Legend for Markers**: The white squares lack explicit l

In [6]:
# ── Full batch ────────────────────────────────────────────────────────────────
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

results = batch_query_images(
    PROMPT,
    image_root=IMAGE_ROOT,
    models=MODELS,
    output_root=OUTPUT_ROOT,
    run_id=RUN_ID,
    system=SYSTEM_PROMPT,
    collection_ids=COLLECTIONS,
    delay=DELAY,
    verbose=True,
)

print(f'\nDone. {len(results)} queries completed.')

[1/6] model=qwen2.5vl:latest  plot=ecalOccRecdEtWgt  image=ecalOccRecdEtWgt_run398185.png ... 8.87s → results/baseline/ecalOccRecdEtWgt/ecalOccRecdEtWgt_run398185_qwen2.5vl_latest.txt
[2/6] model=qwen2.5vl:32b  plot=ecalOccRecdEtWgt  image=ecalOccRecdEtWgt_run398185.png ... 54.03s → results/baseline/ecalOccRecdEtWgt/ecalOccRecdEtWgt_run398185_qwen2.5vl_32b.txt
[3/6] model=qwen3-vl:latest  plot=ecalOccRecdEtWgt  image=ecalOccRecdEtWgt_run398185.png ... ERROR
[4/6] model=litellm-ow.qwen/qwen3.6  plot=ecalOccRecdEtWgt  image=ecalOccRecdEtWgt_run398185.png ... 50.63s → results/baseline/ecalOccRecdEtWgt/ecalOccRecdEtWgt_run398185_litellm-ow.qwen_qwen3.6.txt
[5/6] model=gemma3:latest  plot=ecalOccRecdEtWgt  image=ecalOccRecdEtWgt_run398185.png ... 8.8s → results/baseline/ecalOccRecdEtWgt/ecalOccRecdEtWgt_run398185_gemma3_latest.txt
[6/6] model=litellm-ow.google/gemma4-31b  plot=ecalOccRecdEtWgt  image=ecalOccRecdEtWgt_run398185.png ... 47.66s → results/baseline/ecalOccRecdEtWgt/ecalOccRecdEt

In [ ]:
# ── Retry errors ──────────────────────────────────────────────────────────────
# Re-runs only queries that errored in the current session's `results` list.
# Merges successful retries back into `results` in-place.

failed = [
    (r['model_used'], r['image'])
    for r in results if r['error'] is not None
]

if not failed:
    print('No errors in results — nothing to retry.')
else:
    print(f'Retrying {len(failed)} failed quer{"y" if len(failed)==1 else "ies"}...')
    retry_results = []

    for i, (model, image_str) in enumerate(failed):
        image_path = Path(image_str)
        plot_name  = image_path.parent.name
        print(f'  [{i+1}/{len(failed)}] model={model}  image={image_path.name} ...', end=' ', flush=True)

        result = query(
            PROMPT,
            model=model,
            system=SYSTEM_PROMPT,
            image_path=image_path,
            collection_ids=COLLECTIONS,
        )
        result['plot_name'] = plot_name
        retry_results.append(result)

        out_dir  = resolve_output_dir(OUTPUT_ROOT, plot_name, RUN_ID)
        out_dir.mkdir(parents=True, exist_ok=True)
        out_file = resolve_output_file(out_dir, image_path, model)
        with open(out_file, 'w') as f:
            f.write(f"Model:    {result['model_used']}\n")
            f.write(f"Plot:     {plot_name}\n")
            f.write(f"Image:    {result['image']}\n")
            f.write(f"Run ID:   {RUN_ID or '(overwrite)'}\n")
            f.write(f"Latency:  {result['latency_s']}s\n")
            f.write(f"Prompt:   {result['prompt']}\n")
            f.write('-' * 60 + '\n')
            if result['error']:
                f.write(f"ERROR: {result['error']}\n")
            else:
                f.write(result['response'] + '\n')

        status = 'ERROR' if result['error'] else f"{result['latency_s']}s → {out_file}"
        print(status)
        time.sleep(DELAY)

    # Merge back into results
    retry_index = {(r['model_used'], r['image']): r for r in retry_results}
    results = [
        retry_index.get((r['model_used'], r['image']), r)
        for r in results
    ]

    still_failing = sum(1 for r in retry_results if r['error'])
    print(f'\nDone. {len(retry_results) - still_failing}/{len(retry_results)} recovered.')
    if still_failing:
        print('Still failing:')
        for r in retry_results:
            if r['error']:
                print(f"  {r['model_used']}  {Path(r['image']).name}  → {r['error']}")

In [11]:
# ── Summary ───────────────────────────────────────────────────────────────────
df = pd.DataFrame(results)
df['image_name'] = df['image'].apply(lambda p: Path(p).name if p else None)

errors = df[df['error'].notna()]
if not errors.empty:
    print(f'WARNING: {len(errors)} failed queries:')
    display(errors[['plot_name', 'model_used', 'image_name', 'error']])
else:
    print('All queries succeeded.')

print()
display(
    df.groupby(['plot_name', 'model_used'])['latency_s']
      .agg(['count', 'mean', 'min', 'max'])
      .round(2)
      .rename(columns={'count': 'n', 'mean': 'avg_s', 'min': 'min_s', 'max': 'max_s'})
)

All queries succeeded.



n  avg_s  min_s  max_s
plot_name        model_used                               
ecalOccRecdEtWgt gemma3:latest      1  13.91  13.91  13.91
                 google/gemma4-31b  1  51.39  51.39  51.39
                 qwen/qwen3.6       1  45.27  45.27  45.27
                 qwen2.5vl:32b      1  38.58  38.58  38.58
                 qwen2.5vl:latest   1  16.89  16.89  16.89
                 qwen3-vl:latest    1  98.53  98.53  98.53

In [12]:
# ── Save CSV next to the run's output folder ──────────────────────────────────
if RUN_ID:
    csv_path = OUTPUT_ROOT / RUN_ID / f'summary_{RUN_ID}.csv'
else:
    csv_path = OUTPUT_ROOT / 'summary.csv'

df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')

# Show result tree
print()
root = OUTPUT_ROOT / RUN_ID if RUN_ID else OUTPUT_ROOT
for item in sorted(root.iterdir()):
    if item.is_dir():
        txts = list(item.glob('*.txt'))
        print(f'  {item.name}/  ({len(txts)} files)')
        for t in sorted(txts):
            print(f'    {t.name}')
    elif item.suffix == '.csv':
        print(f'  {item.name}')

Saved: results/baseline/summary_baseline.csv

  ecalOccRecdEtWgt/  (6 files)
    ecalOccRecdEtWgt_run398185_gemma3_latest.txt
    ecalOccRecdEtWgt_run398185_litellm-ow.google_gemma4-31b.txt
    ecalOccRecdEtWgt_run398185_litellm-ow.qwen_qwen3.6.txt
    ecalOccRecdEtWgt_run398185_qwen2.5vl_32b.txt
    ecalOccRecdEtWgt_run398185_qwen2.5vl_latest.txt
    ecalOccRecdEtWgt_run398185_qwen3-vl_latest.txt
  summary_baseline.csv


In [13]:
# ── Side-by-side comparison: one image across all models ─────────────────────
COMPARE_PLOT  = plot_names[0]
COMPARE_IMAGE = pairs[0][1].name

subset = df[(df['plot_name'] == COMPARE_PLOT) & (df['image_name'] == COMPARE_IMAGE)]
for _, row in subset.iterrows():
    print('=' * 72)
    print(f"Model   : {row['model_used']}")
    print(f"Latency : {row['latency_s']}s")
    print()
    print(row['error'] and f"ERROR: {row['error']}" or row['response'])
    print()

Model   : qwen2.5vl:latest
Latency : 16.89s

### Instructions for the Input Plot:
The plot is titled "ECal TP ET-weighted Occupancy at Layer1". It shows the occupancy of ECal (Electromagnetic Calorimeter) at Layer1, weighted by the energy transverse (ET) of the deposited energy. The x-axis represents the iEta (eta index), and the y-axis represents the iPhi (phi index). The color bar on the right indicates the number of entries, with a range from 0 to 80,000.

### Description of the Input Plot:
The plot is a heatmap representing the occupancy of ECal at Layer1, with the energy transverse (ET) weight. The x-axis is labeled as iEta, ranging from approximately -25 to 25, and the y-axis is labeled as iPhi, ranging from 0 to 70. The color intensity varies from dark blue (low occupancy) to red (high occupancy), with a legend on the right indicating the number of entries. The plot shows a clear pattern where the occupancy is higher in certain regions, particularly in the central part of the pl